---
**Variational Quantum Eigensolver (VQE).**
VQE is a hybrid quantum-classical algorithm for finding the minimum eigenvalue
(ground state energy) of a Hamiltonian operator $\hat{H}$.
It is described as hybrid because the quantum computer evaluates expectation
values $\langle\psi(\vec{\theta})|\hat{H}|\psi(\vec{\theta})\rangle$
while a classical optimizer iteratively adjusts the circuit parameters
$\vec{\theta}$ to minimize them.

The algorithm rests on the **variational principle**:

$$E_0 \leq \langle\psi(\vec{\theta})|\hat{H}|\psi(\vec{\theta})\rangle
\quad \text{for any } |\psi(\vec{\theta})\rangle$$

where $E_0$ is the true ground state energy.
By minimizing the expectation value over all parameter settings,
VQE converges to (or near) $E_0$.

**Why this Hamiltonian.**
The Hamiltonian used here is a synthetic 2-qubit spin model chosen for
pedagogical reasons rather than derived from a specific physical system.
In quantum computing, Hamiltonians are expressed as weighted sums of
tensor products of Pauli matrices ($I$, $X$, $Y$, $Z$) because
these form a complete basis for all $2^n \times 2^n$ Hermitian operators:

$$\hat{H} = 0.5\,XX + 0.5\,ZZ - 1.0\,IZ - 0.5\,ZI$$

The physical interpretation of each term:
- $XX$ and $ZZ$: qubit-qubit coupling (the $XX$ term drives spin-flip
  transitions, the $ZZ$ term is an Ising-type interaction)
- $IZ$ and $ZI$: local magnetic fields acting on q0 and q1 independently,
  with different strengths ($-1.0$ and $-0.5$) breaking the symmetry
  between the two sites

This Hamiltonian is designed to have three properties that make it
an ideal VQE teaching example:
1.) Its ground state is **entangled**, so the ansatz must include
    entangling gates - a product-state ansatz cannot find $E_0$.
2.) All matrix entries are **real**, so $R_y$ rotations in the ansatz
    are sufficient without needing complex phases.
3.) The exact ground state energy $E_0 \approx -1.20711$ is small enough
    that a shallow circuit with 8 parameters can reach it reliably.

**The three ingredients:**

1.) **Hamiltonian $\hat{H}$** - expressed as a `SparsePauliOp`, a weighted
sum of tensor products of Pauli matrices.

2.) **Ansatz** - a parameterized quantum circuit $|\psi(\vec{\theta})\rangle$
that defines the family of trial states. Here we use `n_local` with
$R_y$ rotation blocks and CZ entanglement blocks on 2 qubits.

3.) **Classical optimizer** - adjusts $\vec{\theta}$ to minimize the
expectation value. Here we use SciPy's COBYLA, a gradient-free optimizer
well suited to noisy objective functions.

This notebook implements VQE from scratch using only core Qiskit
(`StatevectorEstimator`, `SparsePauliOp`, `n_local`) and SciPy
(`scipy.optimize.minimize`), with no dependency on the separate
`qiskit_algorithms` package.
The exact ground state energy can be verified analytically as the minimum
eigenvalue of the $4\times 4$ Hamiltonian matrix:
$E_0 \approx -1.20711$.

---
**Cell 01 - VQE implemented from scratch.**

**Why this ansatz.**
`n_local(2, rotation_blocks='ry', entanglement_blocks='cz')` builds a
layered circuit with alternating $R_y(\theta)$ rotation layers and CZ
entanglement layers, repeated for `reps=3` (the default), giving
$2 \times (3+1) = 8$ free parameters.
$R_y$ rotations are chosen because they generate real-valued amplitudes,
which is sufficient to reach the ground state of this particular Hamiltonian
(all matrix entries are real).
CZ entanglement is necessary because the ground state is entangled -
no product state can reach $E_0$.
The ansatz must be expressive enough to contain the true ground state
within its parameter space, while remaining shallow enough to be tractable.

**What happens at each iteration.**
COBYLA (Constrained Optimization BY Linear Approximation) is a
gradient-free optimizer that builds a local linear model of the cost
function from function evaluations alone - no gradient is computed.
At each call to `__call__`, the current parameter vector is bound
to the ansatz, `StatevectorEstimator` computes the exact expectation value
$\langle\psi(\vec{\theta})\,|\,\hat{H}\,|\,\psi(\vec{\theta})\rangle$,
and that scalar is returned to COBYLA as the objective.
COBYLA then updates its linear model and proposes a new parameter vector
that it expects to lower the objective.

**Why it takes many iterations.**
With 8 parameters and a non-convex energy landscape full of local minima,
COBYLA typically needs several hundred evaluations to converge.
The `rhobeg=0.5` setting controls the initial trust-region radius -
larger values explore more aggressively at the start but may overshoot
near the minimum.
A random initial point `x0` is used with a fixed seed for reproducibility;
a poor starting point can require more iterations or land in a local minimum.

**Why `cost()` is a method rather than a standalone function.**
`scipy.optimize.minimize` expects a callable `f(params) -> float`
with no other arguments.
Rather than using global variables or passing extra arguments via a closure,
the cost function is implemented as the `__call__` method of `VQEObjective`.

In Python, `__call__` is a **dunder** (double-underscore) method that makes
any class instance behave like a function: when you write `obj(params)`,
Python automatically invokes `obj.__call__(params)`.
This means the optimizer can treat `obj` exactly as if it were a plain
function, while `obj` privately carries the ansatz, Hamiltonian, and
estimator as instance attributes set in `__init__`.
The result is a self-contained, reusable object with no global state
and no closure over outer-scope variables.

In [ ]:
"""vqe.ipynb"""

# Cell 01 - VQE from scratch: find the ground state energy of a 2-qubit Hamiltonian
# Uses only core Qiskit + SciPy - no qiskit_algorithms dependency

import numpy as np
from IPython.display import Math, display
from qiskit.circuit.library import n_local
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from scipy.optimize import minimize

from qis101_utils import as_latex


class VQEObjective:
    """Callable cost function for VQE optimization.

    Stores the ansatz, Hamiltonian, and estimator so that
    scipy.optimize.minimize receives a clean f(params)->float
    callable with no global variables or extra arguments.

    Parameters
    ----------
    ansatz : QuantumCircuit
        Parameterized trial circuit.
    hamiltonian : SparsePauliOp
        The operator whose expectation value is minimized.
    estimator : StatevectorEstimator
        V2 primitive for exact expectation value evaluation.
    """

    def __init__(self, ansatz, hamiltonian, estimator):
        self.ansatz = ansatz
        self.hamiltonian = hamiltonian
        self.estimator = estimator

    def __call__(self, params: np.ndarray) -> float:
        """Evaluate <psi(params)|H|psi(params)> for the given parameter vector."""
        bound = self.ansatz.assign_parameters(params)
        job = self.estimator.run([(bound, self.hamiltonian)])
        return float(job.result()[0].data.evs.real)


# Define the Hamiltonian as a weighted sum of Pauli operators
# Qiskit ordering: rightmost Pauli acts on q0
H = SparsePauliOp.from_list(
    [
        ("XX", 0.5),  # 0.5 * X(q1) x X(q0)
        ("ZZ", 0.5),  # 0.5 * Z(q1) x Z(q0)
        ("IZ", -1.0),  # -1.0 * I(q1) x Z(q0)
        ("ZI", -0.5),  # -0.5 * Z(q1) x I(q0)
    ]
)

# Ansatz: 2 qubits, Ry rotations, CZ entanglement, default reps=3 -> 8 parameters
ansatz = n_local(2, rotation_blocks="ry", entanglement_blocks="cz")

# Build the callable cost object and run the classical optimizer
obj = VQEObjective(ansatz, H, StatevectorEstimator())

rng = np.random.default_rng(seed=42)
x0 = rng.uniform(-np.pi, np.pi, ansatz.num_parameters)
opt_result = minimize(
    obj, x0, method="COBYLA", options={"maxiter": 1000, "rhobeg": 0.5}
)

vqe_energy = opt_result.fun
exact_energy = float(np.min(np.linalg.eigvalsh(H.to_matrix())))
rel_error = abs(vqe_energy - exact_energy) / abs(exact_energy) * 100

display(as_latex(H.to_matrix(), prefix=r"\mathbf{\hat{H}}="))
display(Math(rf"\text{{VQE ground state energy : }}{vqe_energy:.6f}"))
display(Math(rf"\text{{Exact ground state energy: }}{exact_energy:.6f}"))
display(Math(rf"\text{{Relative error           : }}{rel_error:.4f}\%"))
display(Math(rf"\text{{Optimizer iterations     : }}{opt_result.nfev}"))


**Cell 02 - Eigenspectrum of the fixed Hamiltonian.**
Before running VQE it is instructive to see what we are hunting for.
The $4\times 4$ Hamiltonian has four exact eigenvalues computed by
`numpy.linalg.eigvalsh`.
Each horizontal line represents one energy level; the ground state
$E_0 \approx -1.2071$ is shown in red.
VQE's job is to find that lowest level using only expectation value
queries - without ever seeing the full matrix or its eigendecomposition.

In [ ]:
# Cell 02 - Eigenspectrum: show all four energy levels of the fixed Hamiltonian

import matplotlib.pyplot as plt

eigenvalues = np.linalg.eigvalsh(H.to_matrix())

fig, ax = plt.subplots(figsize=(4, 5))
for k, e in enumerate(eigenvalues):
    color = "crimson" if k == 0 else "steelblue"
    lw = 3 if k == 0 else 2
    ax.hlines(e, 0.2, 0.8, colors=color, linewidth=lw)
    ax.text(0.85, e, rf"$E_{k} = {e:.4f}$", va="center", fontsize=10, color=color)

ax.axhline(eigenvalues[0], color="crimson", linestyle="--", linewidth=0.8, alpha=0.4)
ax.set_xlim(0, 1.6)
ax.set_ylabel("Energy (a.u.)")
ax.set_title(r"Eigenspectrum of $\hat{H}$")
ax.set_xticks([])
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
print(f"VQE target: E0 = {eigenvalues[0]:.6f}")


**Cell 03 - Parameterized Hamiltonian: a cosine potential sweep.**
To produce a plot analogous to a molecular potential energy surface,
we introduce a free parameter $\phi$ that continuously varies the
coupling between the two qubits:

$$\hat{H}(\phi) = \cos(\phi)\,XX + \sin(\phi)\,ZZ - IZ - 0.5\,ZI$$

As $\phi$ sweeps from $0$ to $2\pi$, the relative weight of the
$XX$ (spin-flip) and $ZZ$ (Ising) couplings oscillates sinusoidally,
tracing out smooth energy curves for all four eigenvalues.
The local field terms $IZ$ and $ZI$ are held fixed.

This is directly analogous to a potential energy surface:
$\phi$ plays the role of a geometric coordinate (like bond length),
and the ground state curve $E_0(\phi)$ shows the minimum energy
of the system at each coupling configuration.

The exact curves are computed by diagonalizing $\hat{H}(\phi)$
at 200 values of $\phi$.
VQE is then run at 16 evenly spaced values and overlaid as dots,
demonstrating that VQE tracks the ground state curve
without ever computing the full eigendecomposition.

In [ ]:
# Cell 03 - Parameterized Hamiltonian: H(phi) = cos(phi)*XX + sin(phi)*ZZ - IZ - 0.5*ZI
# Sweep phi from 0 to 2*pi and compute exact eigenvalues + VQE estimates

from qiskit.quantum_info import SparsePauliOp


def make_hamiltonian(phi: float) -> SparsePauliOp:
    """Build the parameterized 2-qubit Hamiltonian at coupling angle phi."""
    return SparsePauliOp.from_list(
        [
            ("XX", np.cos(phi)),
            ("ZZ", np.sin(phi)),
            ("IZ", -1.0),
            ("ZI", -0.5),
        ]
    )


# Exact eigenvalue curves at 200 phi values
phi_dense = np.linspace(0, 2 * np.pi, 200)
exact_energies = np.array(
    [np.linalg.eigvalsh(make_hamiltonian(phi).to_matrix()) for phi in phi_dense]
)

# VQE estimates at 16 phi values
phi_vqe = np.linspace(0, 2 * np.pi, 16, endpoint=False)
vqe_energies = []

estimator = StatevectorEstimator()
ansatz_sweep = n_local(2, rotation_blocks="ry", entanglement_blocks="cz")
rng = np.random.default_rng(seed=42)
x0 = rng.uniform(-np.pi, np.pi, ansatz_sweep.num_parameters)

print("Running VQE at 16 phi values...")
for i, phi in enumerate(phi_vqe):
    H_phi = make_hamiltonian(phi)
    obj = VQEObjective(ansatz_sweep, H_phi, estimator)
    result = minimize(
        obj, x0, method="COBYLA", options={"maxiter": 1000, "rhobeg": 0.5}
    )
    vqe_energies.append(result.fun)
    # Warm-start: use optimal params as initial guess for next phi
    x0 = result.x
    e_exact = np.min(np.linalg.eigvalsh(make_hamiltonian(phi).to_matrix()))
    e_vqe = result.fun
    rel_err = abs(e_vqe - e_exact) / abs(e_exact) * 100
    print(
        f"  phi={phi:5.3f}  E0_vqe={e_vqe:9.5f}  E0_exact={e_exact:9.5f}  rel_err={rel_err:.4f}%"
    )

vqe_energies = np.array(vqe_energies)

# Plot exact curves + VQE dots
colors = ["crimson", "steelblue", "seagreen", "darkorange"]
labels = [rf"$E_{k}$ (exact)" for k in range(4)]

fig, ax = plt.subplots(figsize=(9, 5))
for k in range(4):
    ax.plot(
        phi_dense, exact_energies[:, k], color=colors[k], linewidth=2, label=labels[k]
    )
ax.scatter(
    phi_vqe,
    vqe_energies,
    color="crimson",
    zorder=5,
    s=60,
    marker="o",
    label=r"$E_0$ (VQE)",
)

ax.set_xlabel(r"Coupling angle $\phi$ (radians)")
ax.set_ylabel("Energy (a.u.)")
ax.set_title(
    r"Eigenspectrum of $\hat{H}(\phi) = \cos\phi\,XX + \sin\phi\,ZZ - IZ - 0.5\,ZI$"
)
ax.set_xticks(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi],
    [r"$0$", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"],
)
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Cell 04 - Why VQE matters: the classical memory wall.**
The 2-qubit Hamiltonian above is trivially solvable classically - Mathematica
finds its minimum eigenvalue instantly with `Min@Eigenvalues[h]`.
But the dimension of the Hilbert space grows as $2^n$ with the number of
qubits $n$, and classical exact diagonalization becomes impossible very quickly.

In computational chemistry, simulating a molecule requires representing its
electronic wavefunction in a basis of atomic orbitals.
Each spatial orbital becomes two spin orbitals (spin-up and spin-down),
and with the Jordan-Wigner transformation each spin orbital maps to one qubit.
The resulting Hilbert space has dimension $2^{n_{\text{qubits}}}$:

| Molecule | Atoms | Qubits (STO-3G) | Classical RAM needed | Accessible to |
|---|---|---|---|---|
| H<sub>2</sub> | 2 | 4 | ~1 KB | Classical |
| LiH | 4 | 12 | ~64 KB | Classical |
| Naphthalene C<sub>10</sub>H<sub>8</sub> | 18 | 116 | ~10<sup>27</sup> bytes | Quantum only |
| Caffeine C<sub>8</sub>H<sub>10</sub>N<sub>4</sub>O<sub>2</sub> | 24 | 160 | ~10<sup>34</sup> bytes | Quantum only |
| Retinal C<sub>20</sub>H<sub>28</sub>O | 49 | 240 | ~10<sup>57</sup> bytes | Quantum only |

To put $10^{34}$ bytes in context: the total amount of data estimated to exist
on Earth in 2025 is roughly $10^{23}$ bytes (100 zettabytes).
Simulating caffeine classically would require $10^{11}$ times more storage
than all data on Earth.

Simulating naphthalene (C<sub>10</sub>H<sub>8</sub>) would require approximately
10<sup>34</sup> bits on a classical computer.

IBM's current Heron processor has 156 qubits - enough in principle to run
VQE for molecules in the caffeine size range, though noise and circuit depth
remain practical challenges.
IBM's roadmap targets a fault-tolerant system capable of running
100 million gates on 200 logical qubits by 2029, at which point exact
simulation of molecules in this range becomes genuinely practical.

---
**Cell 04** below constructs the Hamiltonian for the hydrogen molecule H<sub>2</sub>
at its equilibrium bond length - the simplest real chemical example -
and runs VQE to find its ground state energy.
This is the canonical VQE benchmark problem first demonstrated by
Google and Harvard in 2016.

**Where the numbers come from.**

The starting point is the Schrödinger equation for the H<sub>2</sub> molecule:
two protons plus two electrons.
The electronic Hamiltonian in atomic units (Hartree) is:

$$\hat{H} = -\sum_i \frac{\nabla_i^2}{2} - \sum_{i,A} \frac{Z_A}{r_{iA}} + \sum_{i<j} \frac{1}{r_{ij}} + \frac{Z_A Z_B}{R_{AB}}$$

The four terms are kinetic energy of electrons, electron-nucleus attraction,
electron-electron repulsion, and nuclear-nuclear repulsion.
This is a continuous function over all space and cannot be put directly
into a quantum computer.

**The STO-3G basis set.**

To discretize it, quantum chemists expand the electron orbitals in a finite
set of basis functions (here STO-3G, the minimal basis).
For H<sub>2</sub> there are 2 atoms × 1 orbital per hydrogen atom = 2 spatial
orbitals, giving 4 spin-orbitals (each spatial orbital has a spin-up and
spin-down version).
This maps the continuous Hamiltonian into a sum of integrals over the basis:

$$\hat{H} = \sum_{pq} h_{pq}\, a_p^\dagger a_q + \sum_{pqrs} g_{pqrs}\, a_p^\dagger a_q^\dagger a_r a_s$$

The $h_{pq}$ coefficients are **one-body integrals** (kinetic energy plus
nuclear attraction for a single electron) and $g_{pqrs}$ are **two-body
integrals** (electron-electron repulsion between pairs).
These are computed numerically by classical quantum chemistry codes such as
PySCF or Psi4 at the chosen bond length (0.735 Å here).
The specific numbers like `-1.052373...` are the results of those
numerical integrals - they are not chosen by hand.

**The Jordan-Wigner transformation.**

The operators $a_p^\dagger$ (create an electron in orbital $p$) and $a_q$
(destroy one) are **fermionic** - they obey anticommutation relations that
quantum computers cannot represent directly.
The Jordan-Wigner transformation maps each fermionic operator to a string
of Pauli matrices:

$$a_p^\dagger = \frac{X_p - iY_p}{2} \otimes Z_{p-1} \otimes Z_{p-2} \otimes \cdots \otimes Z_0$$

The chain of $Z$ gates before position $p$ tracks the fermionic sign
(the anticommutation).
After applying this transformation to every $a_p^\dagger a_q$ and
$a_p^\dagger a_q^\dagger a_r a_s$ term and simplifying algebraically,
the Hamiltonian becomes exactly the sum of weighted Pauli strings in the code.
The coefficients are inherited directly from the $h_{pq}$ and $g_{pqrs}$
integrals.

**What each Pauli string means physically.**

- `IIII` ($-1.052\ldots$): constant offset - nuclear repulsion energy plus
  the part of the one-body energy that factors out. Every eigenvalue is
  shifted by this amount.
- `IIIZ`, `IIZI`, `IZII`, `ZIII` ($\pm 0.397\ldots$): one-body terms
  describing a single electron moving in the field of both nuclei.
  The alternating signs arise from how the Jordan-Wigner $Z$ strings
  interact with the orbital ordering.
- `IIZZ`, `IZIZ`, `ZIIZ`, `ZZII` ($\pm 0.011\ldots$): two-body **Coulomb**
  terms - the direct electrostatic repulsion between electron pairs
  occupying specific orbital combinations.
  The small coefficient reflects that this is a weak correction at
  equilibrium bond length.
- `IXIX`, `IXZX`, `IYIY`, `IYZY` ($\pm 0.180\ldots$): two-body **exchange**
  terms arising from the antisymmetry of the fermionic wavefunction.
  These have no classical analogue - they are purely quantum mechanical
  and represent the energy contribution from electrons being
  indistinguishable.

**Why these particular Pauli strings.**

The Pauli strings are the mathematical output of the
Jordan-Wigner transformation applied to the chemistry integrals.
A different fermion-to-qubit mapping (such as Bravyi-Kitaev or parity
mapping) would produce a different but equivalent set of Pauli strings,
often with fewer terms or shorter strings, at the cost of a less
transparent qubit-to-orbital correspondence.

In the Jordan-Wigner encoding used here each qubit directly represents
one spin-orbital:

| Qubit | Spin-orbital |
|---|---|
| 0 | Orbital 0, spin-up |
| 1 | Orbital 0, spin-down |
| 2 | Orbital 1, spin-up |
| 3 | Orbital 1, spin-down |

In [ ]:
# Cell 04 - VQE for the H2 molecule (the canonical quantum chemistry benchmark)
#
# The H2 Hamiltonian in the STO-3G minimal basis at equilibrium bond length
# (0.735 Angstrom) after the Bravyi-Kitaev transformation uses 4 qubits.
# This Hamiltonian for DiHydrogen was derived from quantum chemistry calculations and
# is the standard first benchmark for any VQE implementation.
# Coefficients from: Peruzzo et al., Nature Communications 5, 4213 (2014)
# - the paper that introduced VQE.
#
# The exact ground state energy at equilibrium is -1.8572 Hartree.
# 1 Hartree = 27.211 eV = 627.5 kcal/mol (chemical energy unit)
# Chemical accuracy threshold: < 1 kcal/mol = 0.0016 Hartree

# H2 Hamiltonian at 0.735 Angstrom bond length (4 qubits, Bravyi-Kitaev encoding)
H2 = SparsePauliOp.from_list(
    [
        ("IIII", -1.052373245772859),  # constant (nuclear repulsion + core energy)
        ("IIIZ", 0.397936724843180),  # one-body term
        ("IIZI", -0.397936724843180),  # one-body term
        ("IIZZ", -0.011280104256235),  # two-body Coulomb
        ("IXIX", 0.180931199784232),  # two-body exchange
        ("IXZX", -0.180931199784232),  # two-body exchange
        ("IYIY", 0.180931199784232),  # two-body exchange
        ("IYZY", -0.180931199784232),  # two-body exchange
        ("IZII", -0.397936724843180),  # one-body term
        ("IZIZ", 0.011280104256235),  # two-body Coulomb
        ("IZZZ", 0.397936724843180),  # one-body term
        ("ZIII", -0.397936724843180),  # one-body term
        ("ZIIZ", 0.011280104256235),  # two-body Coulomb
        ("ZIZI", -0.397936724843180),  # one-body term
        ("ZZII", 0.011280104256235),  # two-body Coulomb
        ("ZZIZ", 0.397936724843180),  # one-body term
        ("ZZZI", -0.397936724843180),  # one-body term
    ]
)

# 4-qubit ansatz: Ry rotations + CZ entanglement
ansatz_h2 = n_local(4, rotation_blocks="ry", entanglement_blocks="cz")

obj_h2 = VQEObjective(ansatz_h2, H2, StatevectorEstimator())
rng_h2 = np.random.default_rng(seed=7)
x0_h2 = rng_h2.uniform(-np.pi, np.pi, ansatz_h2.num_parameters)

opt_h2 = minimize(
    obj_h2, x0_h2, method="COBYLA", options={"maxiter": 2000, "rhobeg": 0.5}
)

vqe_h2 = opt_h2.fun
exact_h2 = float(np.min(np.linalg.eigvalsh(H2.to_matrix())))
rel_err = abs(vqe_h2 - exact_h2) / abs(exact_h2) * 100
chem_acc = abs(vqe_h2 - exact_h2) * 627.5  # convert Hartree to kcal/mol

display(
    Math(
        rf"\text{{H}}_2\text{{ VQE ground state energy : }}{vqe_h2:.6f}\text{{ Hartree}}"
    )
)
display(
    Math(
        rf"\text{{H}}_2\text{{ exact ground state energy: }}{exact_h2:.6f}\text{{ Hartree}}"
    )
)
display(Math(rf"\text{{Relative error              : }}{rel_err:.4f}\%"))
display(
    Math(
        rf"\text{{Error in kcal/mol            : }}{chem_acc:.4f}"
        rf"\quad(\text{{chemical accuracy}} < 1.0)"
    )
)
display(Math(rf"\text{{Optimizer iterations         : }}{opt_h2.nfev}"))
